# 1. Data


In [ ]:
import numpy as np
import importlib
from data.data_reader import Data
%load_ext autoreload
%autoreload 2

# from preprocessing import pre
Data_params_file = 'Ensemble_configs/Data_params.py'
loader = importlib.machinery.SourceFileLoader('Data_params', Data_params_file)
Data_params = loader.load_module()

In [ ]:
data = Data(**Data_params.data_base)

In [ ]:
x_train, x_test, y_train, y_test = data.get_train_test()

In [ ]:
data.columns.levels[0]

# 2. Network

In [ ]:
from network.network_builder import get_tissue_specific_maps
import pandas as pd

In [ ]:
# select tissues name from gtex and append it in Network_params.tissues list 
Network_params_file = 'Ensemble_configs/Network_params.py'
loader = importlib.machinery.SourceFileLoader('Network_params', Network_params_file)
Network_params = loader.load_module()

In [ ]:
# TIME-CONSUMING  ALERT!!!!!!!!!!!!! 
# insted of runing this cell multiple times,
# save the tissue_maps and load for next time trainig
tissue_maps = get_tissue_specific_maps(genes=data.columns.levels[0],
                                       Network_params=Network_params)

In [ ]:
# for the first time save it 
# import pickle
# with open('tissue_maps.pkl', 'wb') as f:
#     pickle.dump(tissue_maps, f)

In [ ]:
# for the second time load it 
import pickle
with open('tissue_maps.pkl', 'rb') as f:
    tissue_maps = pickle.load(f)

# 3. Model

## 3.1 Building simple model

In [ ]:
from model.model_builder import * 
from keras.utils import plot_model

In [ ]:
Model_params_file = 'Ensemble_configs/Model_params.py'
loader = importlib.machinery.SourceFileLoader('Model_params', Model_params_file)
Model_params = loader.load_module()

In [ ]:
tissue_maps.keys()

In [ ]:
maps_1 = tissue_maps['Prostate']
# maps_2 = tissue_maps['Testis']

In [ ]:
model_1 = build_pnet(data=data, maps=maps_1, **Model_params.model_params)
# model_2 = build_pnet(data=data, maps=maps_2, **Model_params.model_params)

In [ ]:
plot_model(model_1, to_file = 'pnet_model_graph.jpg',expand_nested=True,show_layer_activations=True, show_shapes = True, show_layer_names = True)

In [ ]:
callbacks = get_callbacks(x_train, y_train)

In [ ]:
history = model_1.fit(x_train,
                    y_train,
                    epochs=60,
                    batch_size=100,
                    verbose=2,
                    callbacks=callbacks)

In [ ]:
# history = model_2.fit(x_train,
#                     y_train,
#                     epochs=300,
#                     batch_size=100,
#                     verbose=2,
#                     callbacks=callbacks)

In [ ]:
y_pred = model_1.predict(x_test)
get_th(y_test, y_pred)

In [ ]:
# y_pred = model_2.predict(x_test)
# get_th(y_test, y_pred)

### LIME

In [ ]:
from lime.lime_tabular import LimeTabularExplainer 
explainer = LimeTabularExplainer(x_train, 
                      feature_names=data.columns, 
                      class_names=['Primary', 'Metastatic'], 
                      discretize_continuous=False, 
                      verbose=True)

In [ ]:
def predict_prob(z):
    z_prob = model_1.predict(z)
    z_prob = np.concatenate([ 1-z_prob, z_prob],axis=1)
    return z_prob

In [ ]:
lime = explainer.explain_instance(x_test[0:10], predict_prob)
lime.show_in_notebook(show_table=True)

### DeepLift

In [ ]:
from keras.models import Sequential
from keras.layers import Dropout, BatchNormalization
import sys

In [ ]:
def get_layers(model, level=1):
    layers = []
    for i, l in enumerate(model.layers):

        # indent = '  ' * level + '-'
        if type(l) == Sequential:
            layers.extend(get_layers(l, level + 1))
        else:
            layers.append(l)

    return layers

In [ ]:
layers = get_layers(model_1)

In [ ]:
def get_deep_explain_scores(model, X_train, y_train, target=-1, method_name='grad*input', detailed=False, **kwargs):
    # gradients_list = []
    # gradients_list_sample_level = []

    gradients_list = {}
    gradients_list_sample_level = {}

    i = 0
    for l in get_layers(model):
        if type(l) in [Sequential, Dropout, BatchNormalization]:
            continue
        if l.name.startswith('h') or l.name.startswith('Genes'):  # hidden layers (this is just a convention )

            if target is None:
                output = i
            else:
                output = target

            print('layer # {}, layer name {},  output name {}'.format(i, l.name, output))
            i += 1
            gradients = get_deep_explain_score_layer(model, X_train, l.name, output, method_name=method_name)
            if gradients.ndim > 1:
                # feature_weights = np.sum(np.abs(gradients), axis=-2)
                # feature_weights = np.sum(gradients, axis=-2)
                print('gradients.shape', gradients.shape)
                # feature_weights = np.abs(np.sum(gradients, axis=-2))
                feature_weights = np.sum(gradients, axis=-2)
                # feature_weights = np.mean(gradients, axis=-2)
                print('feature_weights.shape', feature_weights.shape)
                print('feature_weights min max', min(feature_weights), max(feature_weights))
            else:
                # feature_weights = np.abs(gradients)
                feature_weights = gradients
                # feature_weights = np.mean(gradients)
            # gradients_list.append(feature_weights)
            # gradients_list_sample_level.append(gradients)
            gradients_list[l.name] = feature_weights
            gradients_list_sample_level[l.name] = gradients
    if detailed:
        return gradients_list, gradients_list_sample_level
    else:
        return gradients_list
    pass

In [ ]:
def get_deep_explain_score_layer(model, X, layer_name, output_index=-1, method_name='grad*input'):
    scores = None
    import keras
    from deepexplain.deepexplain.tensorflow.methods import DeepExplain
    import tensorflow as tf
    ww = model.get_weights()
    with tf.compat.v1.Session() as sess:
        try:
            with DeepExplain(session=sess) as de:  # <-- init DeepExplain context
                # Need to reconstruct the graph in DeepExplain context, using the same weights.
                # model= nn_model.model
                print(layer_name)
                model = keras.models.clone_model(model)
                model.set_weights(ww)
                # if layer_name=='inputs':
                #     layer_outcomes = X
                # else:
                #     layer_outcomes = nn_model.get_layer_output(layer_name, X)[0]

                x = model.get_layer(layer_name).output
                # x = model.inputs[0]
                if type(output_index) == str:
                    y = model.get_layer(output_index).output
                else:
                    y = model.outputs[output_index]

                # y = model.get_layer('o6').output
                # x = model.inputs[0]
                print(layer_name)
                print('model.inputs', model.inputs)
                print('model y', y)
                print('model x', x)
                attributions = de.explain(method_name, y, x, model.inputs[0], X)
                print('attributions', attributions.shape)
                scores = attributions
                return scores
        except:
            sess.close()
            print(("Unexpected error:", sys.exc_info()[0]))
            raise

In [ ]:
coef_ = get_deep_explain_scores(model_1, x_train, y_train,
                                target=-1, method_name='deeplift',
                                detailed=False)

## 3.2 Building Ensemble model

In [ ]:
from model.ensemble_model_builder import *
from keras.utils import plot_model

In [ ]:
Model_params_file = 'Ensemble_configs/Model_params.py'
loader = importlib.machinery.SourceFileLoader('Model_params', Model_params_file)
Model_params = loader.load_module()

In [ ]:
tissue_maps.keys()

In [ ]:
data.columns

In [ ]:
model = build_ensemble_pnet(data=data, tissue_maps=tissue_maps, **Model_params.model_params)

In [ ]:
plot_model(model, to_file = 'pnet_ensemble_model_graph.jpg',expand_nested=True,show_layer_activations=True, show_shapes = True, show_layer_names = True)

In [ ]:
callbacks = get_callbacks(x_train, y_train)

In [ ]:
history = model.fit(x_train,
                    y_train,
                    epochs=300,
                    batch_size=100,
                    verbose=2,
                    callbacks=callbacks)

In [ ]:
y_pred = model.predict(x_test)
get_th(y_test, y_pred)